# LeetCode 4078 — Three Fibonacci Sum bằng Python

Link đề: https://leetcode.com/problems/three-fibonacci-sum/ · Độ khó: **Easy** · Chủ đề: Math, Hash Table, Fibonacci

Skeleton LeetCode:

```python
class Solution:
    def threeFibonacciSum(self, n: int) -> bool:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Weekly Contest 523, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập liệt kê **mọi** tổng ba số Fibonacci liên tiếp rồi so sánh với toàn bộ `n` từ `1` đến `3000` và nhiều giá trị ngẫu nhiên lớn tới `10⁹`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dãy Fibonacci `0, 1, 1, 2, 3, 5, 8, 13, ...`; hỏi `n` có phải là tổng của **ba số hạng liên tiếp** của dãy này không.
2. Biết **sinh dãy Fibonacci** bằng phép gán song song `a, b = b, a + b`, và hiểu vì sao với `n ≤ 10⁹` chỉ có khoảng **44 số Fibonacci** cần xét (dãy tăng theo cấp số nhân), nên **không** được duyệt từ `1` đến `n`.
3. Làm được cách **brute force** (sinh dãy rồi thử mọi cửa sổ ba số liên tiếp), cách **cửa sổ trượt** và rút ra **bước đột phá**: `F(i) + F(i+1) + F(i+2) = 2·F(i+2)`, nên `n` thỏa mãn khi và chỉ khi **`n` chẵn và `n/2` là số Fibonacci**.
4. Kiểm tra "là số Fibonacci" bằng **tập hợp (`set`) tính trước** hoặc bằng **kiểm tra số chính phương** `5m² ± 4` với `math.isqrt`.
5. Tránh các lỗi kinh điển: dùng **ba số Fibonacci bất kỳ** (không liên tiếp), quên kiểm tra `n` chẵn, thiếu số `1` đầu dãy (mất trường hợp `n = 2`), và sinh dãy không đủ lớn.

## 2. Đề bài & ví dụ

Cho số nguyên `n`.

Dãy Fibonacci bắt đầu bằng `0` và `1`, mỗi số hạng sau là tổng của hai số hạng liền trước: `0, 1, 1, 2, 3, 5, 8, 13, ...`.

Trả về `true` nếu `n` có thể biểu diễn thành **tổng của ba số hạng liên tiếp** trong dãy Fibonacci, ngược lại trả về `false`.

**Ví dụ 1**

- Input: `n = 16`
- Output: `true`
- Giải thích: ba số hạng liên tiếp `3, 5, 8` có tổng `3 + 5 + 8 = 16`.

**Ví dụ 2**

- Input: `n = 8`
- Output: `false`
- Giải thích: tuy `1 + 2 + 5 = 8` nhưng các số hạng đó **không liên tiếp** trong dãy. Không có ba số hạng liên tiếp nào có tổng `8`.

**Ví dụ 3**

- Input: `n = 2`
- Output: `true`
- Giải thích: ba số hạng đầu tiên `0 + 1 + 1 = 2`.

**Ràng buộc:** `1 <= n <= 10⁹`.

In [1]:
from typing import List
import random
from math import isqrt

examples = [
    (16, True),
    (8, False),
    (2, True),
]
for n, expected in examples:
    print(f"n={n}, kỳ vọng {expected}")

n=16, kỳ vọng True
n=8, kỳ vọng False
n=2, kỳ vọng True


## 3. Kiến thức nền

### 3.1. Dãy Fibonacci và phép gán song song `a, b = b, a + b`

Hai số hạng đầu là `0` và `1`; mỗi số sau bằng tổng hai số liền trước. Trong Python, `a, b = b, a + b` gán **đồng thời**: vế phải được tính xong (với `a`, `b` cũ) rồi mới gán. Nhờ vậy ta sinh dãy chỉ với hai biến.

In [2]:
a, b = 0, 1
seq = []
for _ in range(12):
    seq.append(a)
    a, b = b, a + b          # tính xong (b, a + b) rồi mới gán cho (a, b)
print(seq)                    # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### 3.2. Dãy tăng rất nhanh: chỉ có khoảng 44 số Fibonacci ≤ 10⁹

Số Fibonacci tăng gần như **gấp 1.618 lần** mỗi bước, nên đạt `10⁹` chỉ sau khoảng 44 số hạng. Do đó thay vì thử `n` đến `10⁹` giá trị, ta chỉ cần thử vài chục số Fibonacci.

In [3]:
LIMIT = 10**9
fibs = [0, 1]
while fibs[-1] + fibs[-2] <= LIMIT:
    fibs.append(fibs[-1] + fibs[-2])
print("số số hạng (tính cả 0 và 1):", len(fibs))        # 45 (F0 .. F44)
print("số hạng lớn nhất <= 10^9:", fibs[-1], "(F%d)" % (len(fibs) - 1))
print("số hạng kế tiếp:", fibs[-1] + fibs[-2], "> 10^9")
print(fibs[:10], "...", fibs[-3:])

số số hạng (tính cả 0 và 1): 45
số hạng lớn nhất <= 10^9: 701408733 (F44)
số hạng kế tiếp: 1134903170 > 10^9
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34] ... [267914296, 433494437, 701408733]


### 3.3. Vòng `while` với điều kiện dừng theo giá trị

Ta không biết trước cần sinh bao nhiêu số hạng, nên dùng `while` với điều kiện theo **giá trị** (dừng khi vượt giới hạn). `for` với `range(...)` cần biết số lần lặp trước.

In [4]:
def fib_up_to(limit):
    """Các số Fibonacci F0, F1, ... không vượt quá limit."""
    result = []
    a, b = 0, 1
    while a <= limit:
        result.append(a)
        a, b = b, a + b
    return result


print(fib_up_to(20))     # [0, 1, 1, 2, 3, 5, 8, 13]
print(fib_up_to(0))      # [0]

[0, 1, 1, 2, 3, 5, 8, 13]
[0]


### 3.4. `set` và kiểm tra thuộc `in`

`set` lưu các giá trị không trùng và kiểm tra `x in s` trong `O(1)` trung bình. Ta đưa các số Fibonacci vào một `set` để hỏi "`m` có phải số Fibonacci không" nhanh chóng.

In [5]:
fib_set = set(fib_up_to(10**9))
print(8 in fib_set, 4 in fib_set, 5 in fib_set, 0 in fib_set)       # True False True True
print(len(fib_set))                                                  # 44: vì F1 = F2 = 1 chỉ lưu một lần

True False True True
44


### 3.5. Chẵn/lẻ với `%` và chia nguyên với `//`

`n % 2 == 0` kiểm tra `n` chẵn. `n // 2` là chia **làm tròn xuống**: với `n` chẵn `n // 2` đúng bằng `n / 2`, nhưng với `n` lẻ thì **mất phần lẻ** (`5 // 2 = 2`), nên chỉ dùng `n // 2` khi đã biết `n` chẵn.

In [6]:
for n in [16, 8, 5, 7, 2]:
    print(f"n={n}: n % 2 = {n % 2}, n // 2 = {n // 2}, n / 2 = {n / 2}")

n=16: n % 2 = 0, n // 2 = 8, n / 2 = 8.0
n=8: n % 2 = 0, n // 2 = 4, n / 2 = 4.0
n=5: n % 2 = 1, n // 2 = 2, n / 2 = 2.5
n=7: n % 2 = 1, n // 2 = 3, n / 2 = 3.5
n=2: n % 2 = 0, n // 2 = 1, n / 2 = 1.0


### 3.6. `math.isqrt` và kiểm tra số chính phương

`math.isqrt(x)` trả về phần nguyên của căn bậc hai của số nguyên `x ≥ 0` (chính xác, không sai số thực). `x` là **số chính phương** khi `isqrt(x) ** 2 == x`.

In [7]:
def is_square(x):
    return x >= 0 and isqrt(x) ** 2 == x


print([x for x in range(30) if is_square(x)])    # [0, 1, 4, 9, 16, 25]
print(isqrt(17), isqrt(16), is_square(10**18), is_square(10**18 + 1))

[0, 1, 4, 9, 16, 25]
4 4 True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Viết các số Fibonacci đầu tiên với chỉ số `i` rồi cộng từng bộ ba **liên tiếp**:

| `i` | `F(i)` | `F(i+1)` | `F(i+2)` | tổng |
|---|---|---|---|---|
| 0 | 0 | 1 | 1 | **2** |
| 1 | 1 | 1 | 2 | **4** |
| 2 | 1 | 2 | 3 | **6** |
| 3 | 2 | 3 | 5 | **10** |
| 4 | 3 | 5 | 8 | **16** |
| 5 | 5 | 8 | 13 | **26** |
| 6 | 8 | 13 | 21 | **42** |

Các tổng là `2, 4, 6, 10, 16, 26, 42, ...`. Ví dụ 1 (`16`) có mặt; ví dụ 3 (`2`) có mặt (hàng đầu); ví dụ 2 (`8`) **không** có mặt trong danh sách. Quan sát: các tổng đều **chẵn**, và nửa của chúng là `1, 2, 3, 5, 8, 13, 21`: **chính là các số Fibonacci**. Đó là gợi ý cho bước đột phá.

In [8]:
fibs = fib_up_to(100)
print("F      :", fibs)
sums = [fibs[i] + fibs[i + 1] + fibs[i + 2] for i in range(len(fibs) - 2)]
print("tổng 3 :", sums)
print("một nửa:", [s // 2 for s in sums])

F      : [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
tổng 3 : [2, 4, 6, 10, 16, 26, 42, 68, 110, 178]
một nửa: [1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


## 5. Cách 1 — Brute Force / baseline: sinh dãy rồi thử mọi bộ ba liên tiếp

**Ý tưởng.** Sinh danh sách `fibs` gồm mọi số Fibonacci cho tới khi vượt `n` (cộng thêm vài số để chắc chắn đủ). Với mỗi chỉ số `i`, tính `fibs[i] + fibs[i+1] + fibs[i+2]`; nếu bằng `n` thì trả `True`. Hết danh sách mà chưa thấy thì `False`.

In [9]:
def three_fib_bruteforce(n: int) -> bool:
    fibs = [0, 1]
    while fibs[-1] <= n:                          # sinh tới khi số hạng cuối vượt n
        fibs.append(fibs[-1] + fibs[-2])
    for i in range(len(fibs) - 2):                # mọi bộ ba liên tiếp (i, i+1, i+2)
        if fibs[i] + fibs[i + 1] + fibs[i + 2] == n:
            return True
    return False


for n, expected in examples:
    print(f"n={n} -> {three_fib_bruteforce(n)} (kỳ vọng {expected})")

n=16 -> True (kỳ vọng True)
n=8 -> False (kỳ vọng False)
n=2 -> True (kỳ vọng True)


**Độ phức tạp (Cách 1).** Số Fibonacci không vượt `n` là `m ≈ log_φ(n)` (`φ ≈ 1.618`). Sinh dãy `O(m)`, duyệt bộ ba `O(m)`: **Time `O(log n)`**, **Space `O(log n)`** cho danh sách `fibs`. Với `n ≤ 10⁹` thì `m ≈ 44`.

**Nút thắt.** Rất nhanh với một lần gọi, nhưng nếu có nhiều truy vấn ta sinh dãy lại mỗi lần. Hơn nữa, cấu trúc toán học của tổng ba số liên tiếp cho phép trả lời bằng **một phép kiểm tra thuộc tập**, không cần duyệt cửa sổ.

## 6. Ý tưởng thuật toán / bước đột phá

**Tổng ba số Fibonacci liên tiếp bằng hai lần số hạng cuối.** Theo định nghĩa, `F(i+2) = F(i) + F(i+1)`. Vậy

```
F(i) + F(i+1) + F(i+2) = F(i+2) + F(i+2) = 2 · F(i+2)
```

Khi `i` chạy từ `0`, `F(i+2)` chạy qua `F(2), F(3), F(4), ... = 1, 2, 3, 5, 8, ...`: **mọi số Fibonacci `≥ 1`**.

Do đó `n` là tổng ba số Fibonacci liên tiếp **khi và chỉ khi** `n = 2·m` với `m` là một số Fibonacci dương, tức:

```
n chẵn  VÀ  n // 2 là số Fibonacci
```

Chỉ còn việc kiểm tra "`m` có phải số Fibonacci" nhanh: bằng `set` các số Fibonacci tính trước (Cách 3), hoặc bằng kiểm tra `5m² + 4` hoặc `5m² − 4` có phải số chính phương (Cách 4).

In [10]:
fibs = fib_up_to(10**6)
for i in range(len(fibs) - 2):
    assert fibs[i] + fibs[i + 1] + fibs[i + 2] == 2 * fibs[i + 2]
print("đã kiểm tra F(i) + F(i+1) + F(i+2) == 2 * F(i+2) cho", len(fibs) - 2, "giá trị i")
print("n=16:", 16 % 2 == 0 and 16 // 2 in set(fibs), "| n=8:", 8 % 2 == 0 and 8 // 2 in set(fibs), "| n=2:", 2 % 2 == 0 and 2 // 2 in set(fibs))

đã kiểm tra F(i) + F(i+1) + F(i+2) == 2 * F(i+2) cho 29 giá trị i
n=16: True | n=8: False | n=2: True


## 7. Các cách tiếp theo

### Cách 2 — Cửa sổ trượt ba số liên tiếp (không lưu danh sách)

**Giải thích.** Giữ ba biến `a, b, c` là ba số hạng liên tiếp, khởi đầu `0, 1, 1` (tổng `2`). Lặp: nếu `a + b + c == n` trả `True`; nếu `a + b + c > n` dừng (các tổng sau còn lớn hơn vì dãy tăng); ngược lại trượt: `a, b, c = b, c, b + c`. Không lưu danh sách nên bộ nhớ `O(1)`.

In [11]:
def three_fib_window(n: int) -> bool:
    a, b, c = 0, 1, 1                      # F0, F1, F2
    while a + b + c <= n:                  # tổng tăng dần: dừng khi vượt n
        if a + b + c == n:
            return True
        a, b, c = b, c, b + c              # trượt cửa sổ sang phải một ô
    return False


for n, expected in examples:
    print(f"n={n} -> {three_fib_window(n)} (kỳ vọng {expected})")

n=16 -> True (kỳ vọng True)
n=8 -> False (kỳ vọng False)
n=2 -> True (kỳ vọng True)


**Độ phức tạp Cách 2:** **Time `O(log n)`** (khoảng 44 lần lặp), **Space `O(1)`**.

### Cách 3 — Dùng đồng nhất thức: `n` chẵn và `n/2` thuộc tập Fibonacci tính trước (cách của lời giải tham khảo)

**Giải thích.** Tính trước một lần `fib_set` gồm mọi số Fibonacci tới `10⁹` (khoảng 44 phần tử). Mỗi truy vấn chỉ cần `n % 2 == 0 and n // 2 in fib_set`.

In [12]:
FIB_SET = set(fib_up_to(10**9))            # tính một lần, dùng cho mọi truy vấn


def three_fib_set(n: int) -> bool:
    return n % 2 == 0 and n // 2 in FIB_SET


for n, expected in examples:
    print(f"n={n} -> {three_fib_set(n)} (kỳ vọng {expected})")

n=16 -> True (kỳ vọng True)
n=8 -> False (kỳ vọng False)
n=2 -> True (kỳ vọng True)


In [13]:
# Code nộp LeetCode cho Cách 3
class Solution:
    FIB = set()

    def threeFibonacciSum(self, n: int) -> bool:
        if not Solution.FIB:                       # tính trước một lần
            a, b = 0, 1
            while a <= 10**9:
                Solution.FIB.add(a)
                a, b = b, a + b
        return n % 2 == 0 and n // 2 in Solution.FIB

**Độ phức tạp Cách 3:** tiền xử lý **`O(log M)`** thời gian và **`O(log M)`** bộ nhớ với `M = 10⁹` (44 phần tử); mỗi truy vấn **`O(1)`** thời gian.

### Cách 4 — Kiểm tra "là số Fibonacci" bằng số chính phương (không cần bảng)

**Giải thích.** Một số nguyên `m ≥ 0` là số Fibonacci khi và chỉ khi **`5m² + 4` hoặc `5m² − 4` là số chính phương** (tính chất đã biết của dãy Fibonacci). Ta kết hợp với đồng nhất thức: `n` chẵn và `is_fibonacci(n // 2)`. Không lưu bảng, chỉ vài phép nhân và `isqrt`.

In [14]:
def is_fibonacci(m: int) -> bool:
    return m >= 0 and (is_square(5 * m * m + 4) or is_square(5 * m * m - 4))


def three_fib_square(n: int) -> bool:
    return n % 2 == 0 and is_fibonacci(n // 2)


print([m for m in range(60) if is_fibonacci(m)])     # [0, 1, 2, 3, 5, 8, 13, 21, 34, 55]
for n, expected in examples:
    print(f"n={n} -> {three_fib_square(n)} (kỳ vọng {expected})")

[0, 1, 2, 3, 5, 8, 13, 21, 34, 55]
n=16 -> True (kỳ vọng True)
n=8 -> False (kỳ vọng False)
n=2 -> True (kỳ vọng True)


**Độ phức tạp Cách 4:** **Time `O(1)`** theo số phép toán với số nguyên cỡ `10¹⁹` (`isqrt` chạy rất nhanh với số nhỏ như vậy), **Space `O(1)`**.

> Cách 3 dễ hiểu và nhanh nhất cho nhiều truy vấn. Cách 4 không cần tiền xử lý. Cách 1 và Cách 2 cho thấy ý tưởng "tổng ba số liên tiếp" mà không cần biết đồng nhất thức.

## 8. TRACE từng bước

Ta theo dõi Cách 2 (cửa sổ trượt) với `n = 16` và `n = 8`, rồi Cách 3 với `n = 8` và `n = 16`.

In [15]:
def trace_window(n):
    print(f"n={n}")
    a, b, c = 0, 1, 1
    step = 0
    while a + b + c <= n:
        total = a + b + c
        print(f"  bước {step}: (a,b,c)=({a},{b},{c}), tổng={total}", "== n -> True" if total == n else "!= n, trượt")
        if total == n:
            return True
        a, b, c = b, c, b + c
        step += 1
    print(f"  tổng kế tiếp {a + b + c} > n, dừng -> False")
    return False


assert trace_window(16) is True
print()
assert trace_window(8) is False

n=16
  bước 0: (a,b,c)=(0,1,1), tổng=2 != n, trượt
  bước 1: (a,b,c)=(1,1,2), tổng=4 != n, trượt
  bước 2: (a,b,c)=(1,2,3), tổng=6 != n, trượt
  bước 3: (a,b,c)=(2,3,5), tổng=10 != n, trượt
  bước 4: (a,b,c)=(3,5,8), tổng=16 == n -> True

n=8
  bước 0: (a,b,c)=(0,1,1), tổng=2 != n, trượt
  bước 1: (a,b,c)=(1,1,2), tổng=4 != n, trượt
  bước 2: (a,b,c)=(1,2,3), tổng=6 != n, trượt
  tổng kế tiếp 10 > n, dừng -> False


In [16]:
def trace_set(n):
    half = n // 2
    print(f"n={n}: n chẵn? {n % 2 == 0}; n // 2 = {half}; {half} thuộc tập Fibonacci? {half in FIB_SET}")
    result = n % 2 == 0 and half in FIB_SET
    if result:
        print(f"  => True, vì n = 2 * {half} = F(i) + F(i+1) + F(i+2) với F(i+2) = {half}")
    else:
        print("  => False")
    return result


assert trace_set(8) is False
assert trace_set(16) is True
assert trace_set(5) is False

n=8: n chẵn? True; n // 2 = 4; 4 thuộc tập Fibonacci? False
  => False
n=16: n chẵn? True; n // 2 = 8; 8 thuộc tập Fibonacci? True
  => True, vì n = 2 * 8 = F(i) + F(i+1) + F(i+2) với F(i+2) = 8
n=5: n chẵn? False; n // 2 = 2; 2 thuộc tập Fibonacci? True
  => False


Bảng tóm tắt (Cách 2, `n = 8`):

| bước | `(a, b, c)` | tổng | so với `n = 8` |
|---|---|---|---|
| 0 | `(0, 1, 1)` | `2` | nhỏ hơn, trượt |
| 1 | `(1, 1, 2)` | `4` | nhỏ hơn, trượt |
| 2 | `(1, 2, 3)` | `6` | nhỏ hơn, trượt |
| 3 | `(2, 3, 5)` | `10` | **lớn hơn** → dừng, trả `False` |

Tổng nhảy từ `6` lên `10`, bỏ qua `8`; vì tổng chỉ tăng nên không thể quay lại bằng `8`. Đúng với đáp án `false`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (đồng nhất thức + tập Fibonacci). Tập được tính **một lần** ở mức module (ngoài class) để mọi lần gọi dùng lại. Đúng class `Solution`, đúng method `threeFibonacciSum(self, n: int) -> bool`.

In [17]:
# Tính trước tập các số Fibonacci không vượt quá 10^9 (khoảng 44 số)
_fib = set()
_a, _b = 0, 1
while _a <= 10**9:
    _fib.add(_a)
    _a, _b = _b, _a + _b


class Solution:
    def threeFibonacciSum(self, n: int) -> bool:
        # F(i) + F(i+1) + F(i+2) = 2 * F(i+2): n phải chẵn và n // 2 là số Fibonacci
        return n % 2 == 0 and n // 2 in _fib


for n, expected in examples:
    assert Solution().threeFibonacciSum(n) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Đồng nhất thức:** theo định nghĩa `F(i+2) = F(i) + F(i+1)`, nên `F(i) + F(i+1) + F(i+2) = F(i+2) + F(i+2) = 2·F(i+2)`.
- **`F(i+2)` phủ mọi số Fibonacci dương:** với `i = 0, 1, 2, ...`, `F(i+2) = 1, 2, 3, 5, 8, ...`, tức mọi giá trị Fibonacci `≥ 1` (giá trị `1` đã nằm trong `F(2)` vì `F(1) = F(2) = 1`). Chỉ có `0 = F(0)` không phải dạng `F(i+2)`, nhưng nó tương ứng với `n = 0`, ngoài miền `n ≥ 1`; tổng nhỏ nhất thật sự là `0 + 1 + 1 = 2`.
- **Chiều thuận:** nếu `n` là tổng ba số hạng liên tiếp thì `n = 2·F(i+2)`: chẵn và `n // 2 = F(i+2)` là số Fibonacci.
- **Chiều ngược:** nếu `n` chẵn và `n // 2 = F(k)` với `k ≥ 2` (vì `n // 2 ≥ 1`, và `1 = F(2)`), đặt `i = k − 2 ≥ 0` thì `n = 2·F(i+2) = F(i) + F(i+1) + F(i+2)`.
- **Cách 2 dừng đúng lúc:** các tổng `2·F(i+2)` tăng thực sự với `i ≥ 1` (và không giảm ở `i = 0` thành `i = 1`: `2 → 4`), nên khi tổng vượt `n` thì mọi tổng sau còn lớn hơn.
- **Cách 4 đúng nhờ tính chất đã biết:** `m` là số Fibonacci ⇔ `5m² ± 4` là số chính phương; ta kiểm chứng bằng `set` trên toàn bộ `m` tới `10⁶` trong phần test.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số cho trước; số số Fibonacci không vượt quá `n` là `m ≈ log_φ n` (với `n ≤ 10⁹`: `m ≈ 44`).

- **Cách 1 (sinh dãy + duyệt bộ ba):** Time `O(log n)`, Space `O(log n)`.
- **Cách 2 (cửa sổ trượt):** Time `O(log n)`, Space `O(1)`.
- **Cách 3 (đồng nhất thức + `set`):** tiền xử lý `O(log M)` và bộ nhớ `O(log M)` (`M = 10⁹`); mỗi truy vấn **`O(1)`**.
- **Cách 4 (kiểm tra số chính phương):** Time `O(1)` (số học trên số nguyên nhỏ), Space `O(1)`.
- Trung bình và xấu nhất như nhau (không có cấu trúc ngẫu nhiên; `set` chỉ ~44 phần tử).
- **Không** thử từng số từ `1` đến `n`: đó là `O(n) = 10⁹` thao tác, quá chậm; thông tin then chốt là dãy tăng theo cấp số nhân.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` và viết một **lời giải độc lập** `oracle`: **liệt kê mọi tổng ba số hạng liên tiếp** của dãy (tới trên `4·10⁹`) rồi tra bảng, không dùng đồng nhất thức `2·F`.

In [18]:
impls = {
    "bruteforce": three_fib_bruteforce,
    "window": three_fib_window,
    "set": three_fib_set,
    "square": three_fib_square,
    "submit": Solution().threeFibonacciSum,
}


def _all_triple_sums(limit=4 * 10**9):
    """Mọi F(i) + F(i+1) + F(i+2) không vượt limit (dựng bằng danh sách, không dùng 2*F)."""
    f = [0, 1]
    while f[-1] < limit:
        f.append(f[-1] + f[-2])
    return {f[i] + f[i + 1] + f[i + 2] for i in range(len(f) - 2) if f[i] + f[i + 1] + f[i + 2] <= limit}


TRIPLE_SUMS = _all_triple_sums()


def oracle(n):
    return n in TRIPLE_SUMS


print("số tổng ba số liên tiếp <= 4e9:", len(TRIPLE_SUMS), "| nhỏ nhất:", sorted(TRIPLE_SUMS)[:8])
for n, expected in examples:
    assert oracle(n) == expected
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n, fn(n), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

số tổng ba số liên tiếp <= 4e9: 45 | nhỏ nhất: [2, 4, 6, 10, 16, 26, 42, 68]
PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [19]:
def check_all(n, expected, label):
    assert oracle(n) == expected, (label, "oracle", oracle(n), expected)
    for name, fn in impls.items():
        got = fn(n)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all(1, False, "n = 1 nhỏ nhất, lẻ")
check_all(2, True, "n = 2 = 0 + 1 + 1")
check_all(3, False, "n lẻ")
check_all(4, True, "n = 4 = 1 + 1 + 2")
check_all(6, True, "n = 6 = 1 + 2 + 3")
check_all(10, True, "n = 10 = 2 + 3 + 5")
check_all(12, False, "n = 12: chẵn nhưng 6 không là Fibonacci")
check_all(8, False, "ví dụ 2: 1 + 2 + 5 không liên tiếp")
check_all(26, True, "n = 26 = 5 + 8 + 13")
check_all(42, True, "n = 42 = 8 + 13 + 21")
check_all(2 * 433494437, True, "tổng lớn nhất <= 1e9: 2 * F(43)")
check_all(10**9, False, "n = 10^9: chẵn, 5 * 10^8 không là Fibonacci")
check_all(10**9 - 1, False, "n = 10^9 - 1: lẻ")
check_all(2 * 433494437 - 2, False, "chẵn, nhưng nửa là F(43) - 1 (không là Fibonacci)")

PASS n = 1 nhỏ nhất, lẻ -> False
PASS n = 2 = 0 + 1 + 1 -> True
PASS n lẻ -> False
PASS n = 4 = 1 + 1 + 2 -> True
PASS n = 6 = 1 + 2 + 3 -> True
PASS n = 10 = 2 + 3 + 5 -> True
PASS n = 12: chẵn nhưng 6 không là Fibonacci -> False
PASS ví dụ 2: 1 + 2 + 5 không liên tiếp -> False
PASS n = 26 = 5 + 8 + 13 -> True
PASS n = 42 = 8 + 13 + 21 -> True
PASS tổng lớn nhất <= 1e9: 2 * F(43) -> True
PASS n = 10^9: chẵn, 5 * 10^8 không là Fibonacci -> False
PASS n = 10^9 - 1: lẻ -> False
PASS chẵn, nhưng nửa là F(43) - 1 (không là Fibonacci) -> False


### 12.2. Tính chất và phản ví dụ

In [20]:
# Tính chất 1: mọi n lẻ đều False
random.seed(4078)
for _ in range(1000):
    n = random.randrange(1, 10**9, 2)
    assert n % 2 == 1
    assert Solution().threeFibonacciSum(n) is False
print("PASS tính chất: mọi số lẻ đều False")

# Tính chất 2: các giá trị True chính xác là 2*F(k), k >= 2 (42 giá trị <= 1e9)
trues = [n for n in range(1, 10**4) if Solution().threeFibonacciSum(n)]
print("các n <= 10^4 trả True:", trues)
assert trues == [2, 4, 6, 10, 16, 26, 42, 68, 110, 178, 288, 466, 754, 1220, 1974, 3194, 5168, 8362]
all_true = sorted(2 * f for f in FIB_SET if f >= 1 and 2 * f <= 10**9)
assert len(all_true) == 42 and all(Solution().threeFibonacciSum(n) for n in all_true)
print("PASS tính chất: có đúng", len(all_true), "giá trị n <= 10^9 trả True, đều có dạng 2*F(k)")

# Tính chất 3: đồng nhất thức F(i) + F(i+1) + F(i+2) = 2*F(i+2) cho mọi i (đã kiểm ở phần 6); và với n True liền kề
assert sorted(TRIPLE_SUMS)[:10] == [2, 4, 6, 10, 16, 26, 42, 68, 110, 178]

# Phản ví dụ 1: cho phép ba số Fibonacci BẤT KỲ (không liên tiếp) cho kết quả khác
def sai_ba_so_bat_ky(n):
    f = fib_up_to(n)
    for x in f:
        for y in f:
            for z in f:
                if x + y + z == n:
                    return True
    return False

print("n=8: sai (bất kỳ):", sai_ba_so_bat_ky(8), "| đúng:", Solution().threeFibonacciSum(8))
assert sai_ba_so_bat_ky(8) is True and Solution().threeFibonacciSum(8) is False

PASS tính chất: mọi số lẻ đều False
các n <= 10^4 trả True: [2, 4, 6, 10, 16, 26, 42, 68, 110, 178, 288, 466, 754, 1220, 1974, 3194, 5168, 8362]
PASS tính chất: có đúng 42 giá trị n <= 10^9 trả True, đều có dạng 2*F(k)
n=8: sai (bất kỳ): True | đúng: False


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [21]:
# (1) Toàn bộ n từ 1 đến 3000
for n in range(1, 3001):
    expected = oracle(n)
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n)
print("PASS toàn bộ n = 1..3000")

# (2) n ngẫu nhiên lớn tới 1e9, và các giá trị 'gần' True (2*F(k) +- 1, +- 2) để thử chạm biên
random.seed(2026)
specials = sorted(2 * f for f in FIB_SET if f >= 1 and 2 * f <= 10**9)
for trial in range(3000):
    n = random.randint(1, 10**9)
    expected = oracle(n)
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n)
for base in specials:
    for delta in (-2, -1, 0, 1, 2):
        n = base + delta
        if 1 <= n <= 10**9:
            expected = oracle(n)
            for name, fn in impls.items():
                assert fn(n) == expected, (name, n)
print("PASS 3000 n ngẫu nhiên và các giá trị sát 2*F(k): mọi cách khớp oracle độc lập")

PASS toàn bộ n = 1..3000
PASS 3000 n ngẫu nhiên và các giá trị sát 2*F(k): mọi cách khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cho phép ba số Fibonacci bất kỳ (không liên tiếp)

In [22]:
def sai_khong_lien_tiep(n):
    f = fib_up_to(n)
    return any(x + y + z == n for x in f for y in f for z in f)


print("n=8 -> sai:", sai_khong_lien_tiep(8), "| đúng:", Solution().threeFibonacciSum(8))
print("n=7 -> sai:", sai_khong_lien_tiep(7), "| đúng:", Solution().threeFibonacciSum(7))

n=8 -> sai: True | đúng: False
n=7 -> sai: True | đúng: False


**Vì sao sai:** đề yêu cầu **ba số hạng liên tiếp** của dãy. Ví dụ 2 nhấn mạnh: `1 + 2 + 5 = 8` nhưng `1, 2, 5` không liên tiếp (giữa `2` và `5` còn có `3`). Nếu cho phép ba số bất kỳ (kể cả lặp) thì rất nhiều `n` thỏa.

### Lỗi 2: Quên kiểm tra `n` chẵn

In [23]:
def sai_quen_chan(n):
    return n // 2 in FIB_SET                  # n // 2 làm tròn xuống khi n lẻ


print("n=5 -> sai:", sai_quen_chan(5), "| đúng:", Solution().threeFibonacciSum(5))
print("n=7 -> sai:", sai_quen_chan(7), "| đúng:", Solution().threeFibonacciSum(7))

n=5 -> sai: True | đúng: False
n=7 -> sai: True | đúng: False


**Vì sao sai:** tổng ba số liên tiếp luôn bằng `2·F` (chẵn). Với `n` lẻ, `n // 2` làm tròn xuống: `5 // 2 = 2` là số Fibonacci nên code sai trả `True`, trong khi `5` lẻ không thể là `2·F`. Phải có `n % 2 == 0`.

### Lỗi 3: Tập Fibonacci thiếu số `1` đầu dãy

In [24]:
def sai_thieu_so_1(n):
    fib_wrong = set()
    a, b = 2, 3                               # bỏ mất 0, 1, 1
    while a <= 10**9:
        fib_wrong.add(a)
        a, b = b, a + b
    return n % 2 == 0 and n // 2 in fib_wrong


print("n=2 -> sai:", sai_thieu_so_1(2), "| đúng:", Solution().threeFibonacciSum(2))

n=2 -> sai: False | đúng: True


**Vì sao sai:** `n = 2 = 0 + 1 + 1` ứng với `n // 2 = 1`, mà `1` là số Fibonacci (`F(1) = F(2) = 1`). Nếu tập không chứa `1` thì ví dụ 3 bị trả `False`. Hãy sinh dãy từ `a, b = 0, 1` (hoặc bắt đầu ít nhất từ `1`).

### Lỗi 4: Sinh dãy không đủ dài (giới hạn quá nhỏ)

In [25]:
def sai_gioi_han_nho(n):
    fib_small = set()
    a, b = 0, 1
    while a <= 10**6:                         # sai: n có thể tới 10^9
        fib_small.add(a)
        a, b = b, a + b
    return n % 2 == 0 and n // 2 in fib_small


big = 2 * 433494437
print(f"n={big} -> sai:", sai_gioi_han_nho(big), "| đúng:", Solution().threeFibonacciSum(big))

n=866988874 -> sai: False | đúng: True


**Vì sao sai:** `n` lên tới `10⁹`, nên `n / 2` tới `5·10⁸`; cần các số Fibonacci tới ít nhất `5·10⁸` (cho an toàn sinh tới `10⁹`). Giới hạn `10⁶` làm các `n` lớn (như `2·F(43)`) bị trả `False`. Luôn đọc ràng buộc rồi đặt giới hạn sinh dãy theo đó.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**tổng của k số liên tiếp của một dãy truy hồi**": hãy **dùng quan hệ truy hồi để rút gọn** (ở đây `F(i) + F(i+1) = F(i+2)` cho `2·F(i+2)`).
- Dãy tăng theo **cấp số nhân** (Fibonacci, lũy thừa của 2...) → chỉ có **O(log giá trị)** phần tử đến giới hạn; đừng duyệt từ `1` đến `n`.
- Cần hỏi "`m` có thuộc dãy không" nhiều lần: dựng **`set` một lần**.
- Có thể kiểm tra Fibonacci mà **không cần bảng**: `5m² ± 4` là số chính phương.
- `n // 2` chỉ đúng nghĩa "một nửa" khi `n` chẵn; luôn kiểm tra chẵn lẻ trước.
- Các bài Fibonacci khác: tổng `k` số liên tiếp = `F(i+k+1) − F(i+1)`; số Fibonacci nhỏ nhất ≥ n; biểu diễn Zeckendorf (LeetCode 1414).

## 15. Biến thể của bài

1. **Tổng của hai số Fibonacci liên tiếp** là gì? (`F(i) + F(i+1) = F(i+2)`: luôn là số Fibonacci.)
2. **Tổng của `k` số Fibonacci liên tiếp** bằng `n`? (kiểm tra bằng cửa sổ trượt hoặc công thức `F(i+k+1) − F(i+1)`.)
3. **`n` có phải số Fibonacci không** (`is_fibonacci`).
4. **Số Fibonacci nhỏ nhất ≥ n**, hoặc đếm số Fibonacci trong đoạn `[l, r]`.
5. **Biểu diễn `n` thành tổng ít nhất các số Fibonacci** (LeetCode 1414, tham lam).

In [26]:
def sum_k_consecutive(n: int, k: int) -> bool:
    """Biến thể 2: n có là tổng của k số Fibonacci liên tiếp không (k >= 1)? Cửa sổ trượt."""
    fibs = fib_up_to(10**10)
    for i in range(len(fibs) - k + 1):
        if sum(fibs[i:i + k]) == n:
            return True
    return False


def smallest_fib_at_least(n: int) -> int:
    """Biến thể 4."""
    a, b = 0, 1
    while a < n:
        a, b = b, a + b
    return a


def min_fibs_to_sum(n: int) -> int:
    """Biến thể 5: ít nhất bao nhiêu số Fibonacci (được lặp) có tổng n? (tham lam, LeetCode 1414)."""
    fibs = sorted(set(fib_up_to(n)), reverse=True)
    count = 0
    for f in fibs:
        if f and n >= f:
            count += n // f
            n %= f
    return count


for n in range(1, 500):
    assert sum_k_consecutive(n, 3) == Solution().threeFibonacciSum(n)
assert sum_k_consecutive(4, 2) is False and sum_k_consecutive(3, 2) is True       # 1 + 2 = 3
assert sum_k_consecutive(7, 4) is True                                              # 1 + 1 + 2 + 3 = 7
assert [smallest_fib_at_least(n) for n in [0, 1, 4, 13, 14]] == [0, 1, 5, 13, 21]
assert min_fibs_to_sum(7) == 2 and min_fibs_to_sum(10) == 2 and min_fibs_to_sum(19) == 3
print("PASS biến thể")

PASS biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh dãy + duyệt bộ ba | thử mọi cửa sổ 3 số liên tiếp | `O(log n)` | `O(log n)` | hiểu đề; oracle |
| 2. Cửa sổ trượt | `a, b, c = b, c, b + c`, dừng khi tổng > n | `O(log n)` | `O(1)` | không cần danh sách |
| 3. Đồng nhất thức + `set` | `n` chẵn và `n // 2` ∈ tập Fibonacci | tiền xử lý `O(log M)`, truy vấn `O(1)` | `O(log M)` | **cách nộp bài**, nhiều truy vấn |
| 4. Số chính phương | `5m² ± 4` là số chính phương | `O(1)` | `O(1)` | không muốn dựng bảng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `fib_up_to(10)`, `5 // 2`, `7 % 2`, `Solution().threeFibonacciSum(26)`, `Solution().threeFibonacciSum(12)`.

**Bài 2 (tính tay).** Liệt kê các tổng ba số Fibonacci liên tiếp nhỏ hơn `200` (bắt đầu `2, 4, 6, 10, ...`) và cho biết `n = 110` và `n = 100` có thỏa không. Giải thích bằng `2·F`.

**Bài 3 (điền TODO).** Hoàn thành `three_fib_todo(n)` theo Cách 2 (cửa sổ trượt).

In [27]:
def three_fib_todo(n):
    a, b, c = 0, 1, 1
    # TODO 1: trong khi a + b + c <= n
    #   TODO 2: nếu a + b + c == n thì trả True
    #   TODO 3: trượt cửa sổ: a, b, c = b, c, b + c
    # TODO 4: nếu hết vòng lặp mà chưa thấy thì trả False
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `three_fib_square2(n)` theo Cách 4 (số chính phương) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `two_fib_sum(n)` trả `True` nếu `n` là tổng của **hai số Fibonacci liên tiếp** (gợi ý: tổng đó chính là một số Fibonacci). (b) Viết `four_fib_sum(n)` trả `True` nếu `n` là tổng của **bốn số Fibonacci liên tiếp** (tìm công thức rút gọn rồi kiểm tra bằng cửa sổ trượt).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: dãy Fibonacci `0, 1, 1, 2, 3, 5, 8`; `//` chia làm tròn xuống; tổng ba số liên tiếp luôn bằng `2·F`. Bài 2: `n` thỏa khi `n` chẵn và `n/2` là số Fibonacci.
- **Mức 2 (vừa):** Bài 1: `[0, 1, 1, 2, 3, 5, 8]`, `2`, `1`, `True` (`13` là Fibonacci), `False` (`6` không là Fibonacci). Bài 2: các tổng `2, 4, 6, 10, 16, 26, 42, 68, 110, 178`; `110 = 2·55` (có), `100 = 2·50` (`50` không là Fibonacci). Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `m = n // 2`; trả `n % 2 == 0 and (is_square(5*m*m + 4) or is_square(5*m*m - 4))`. Bài 5(a): `F(i) + F(i+1) = F(i+2)`, nên chỉ cần kiểm tra `is_fibonacci(n)` với `n ≥ 1`. Bài 5(b): tổng `k` số liên tiếp `F(i) + ... + F(i+k−1) = F(i+k+1) − F(i+1)`, nên với `k = 4` là `F(i+5) − F(i+1)`; cách đơn giản hơn là dùng cửa sổ trượt kích thước 4 như Cách 2.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert fib_up_to(10) == [0, 1, 1, 2, 3, 5, 8]
assert 5 // 2 == 2 and 7 % 2 == 1
assert Solution().threeFibonacciSum(26) is True
assert Solution().threeFibonacciSum(12) is False

# Bài 2
sums_under_200 = sorted(s for s in TRIPLE_SUMS if s < 200)
assert sums_under_200 == [2, 4, 6, 10, 16, 26, 42, 68, 110, 178]
assert Solution().threeFibonacciSum(110) is True and Solution().threeFibonacciSum(100) is False
assert 110 == 2 * 55 and 55 in FIB_SET and 50 not in FIB_SET

# Bài 3
def three_fib_todo_ans(n):
    a, b, c = 0, 1, 1
    while a + b + c <= n:
        if a + b + c == n:
            return True
        a, b, c = b, c, b + c
    return False

for n in range(1, 500):
    assert three_fib_todo_ans(n) == Solution().threeFibonacciSum(n)

# Bài 4
def three_fib_square2(n):
    m = n // 2
    return n % 2 == 0 and (is_square(5 * m * m + 4) or is_square(5 * m * m - 4))

for n, expected in examples:
    assert three_fib_square2(n) == expected
for n in range(1, 2000):
    assert three_fib_square2(n) == Solution().threeFibonacciSum(n)

# Bài 5(a)
def two_fib_sum(n):
    return n in FIB_SET and n >= 1

assert two_fib_sum(3) and two_fib_sum(5) and not two_fib_sum(4)
assert all(two_fib_sum(a + b) for a, b in zip(fib_up_to(10**6), fib_up_to(10**6)[1:]) if a + b >= 1)

# Bài 5(b): cửa sổ trượt kích thước 4
def four_fib_sum(n):
    f = fib_up_to(10**10)
    return any(f[i] + f[i + 1] + f[i + 2] + f[i + 3] == n for i in range(len(f) - 3))

assert four_fib_sum(4) is True          # 0 + 1 + 1 + 2
assert four_fib_sum(7) is True          # 1 + 1 + 2 + 3
assert four_fib_sum(11) is True         # 1 + 2 + 3 + 5
assert four_fib_sum(12) is False
assert four_fib_sum(18) is True         # 2 + 3 + 5 + 8
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng ba số liên tiếp của dãy truy hồi = hai lần số hạng cuối)

```python
_fib = set()
_a, _b = 0, 1
while _a <= 10**9:
    _fib.add(_a)
    _a, _b = _b, _a + _b


class Solution:
    def threeFibonacciSum(self, n: int) -> bool:
        return n % 2 == 0 and n // 2 in _fib
```

**Checklist**

- [ ] Ba số hạng phải **liên tiếp** (không phải ba số Fibonacci bất kỳ)?
- [ ] Dùng `F(i) + F(i+1) + F(i+2) = 2·F(i+2)`: `n` **chẵn** và `n // 2` là số Fibonacci?
- [ ] Dãy bắt đầu `0, 1` (có `F(1) = F(2) = 1`), để `n = 2` đúng?
- [ ] Giới hạn sinh dãy đủ lớn (≥ `10⁹`, chỉ khoảng 44 số)?
- [ ] Không duyệt từ `1` đến `n`?
- [ ] Đã thử: `n = 1, 2, 8, 16`, số lẻ, giá trị lớn `10⁹`?
- [ ] Đã đối chiếu với liệt kê mọi tổng ba số liên tiếp trên toàn bộ `n` nhỏ?
- [ ] Skeleton đúng `class Solution` / `def threeFibonacciSum(self, n: int) -> bool`?